# Compact Multi-Layer Perceptron (MLP) Baseline for HIGGS Classification

**Author:** Rachit Gupta (neural baseline lead) | **Project:** Higgs Event Lab | **Module:** MID-04

This notebook implements and demonstrates the compact neural network baseline for classifying simulated particle collision events in the HIGGS benchmark. It investigates whether regularized deep neural architectures can effectively learn representations from low-level particle kinematic measurements compared to high-level physicist-engineered invariant masses.

> **Scientific boundaries:** All evaluations in this study classify simulated events generated by Monte Carlo tools. Model outputs represent classification scores on benchmark events, not physical discovery claims.

## Download the original HIGGS data

This cell downloads the original UCI HIGGS dataset when it is missing and builds the processed 250,000 row sample used by the rest of the notebook. It checks for existing files first, so it does not repeat a download. The source link is the official UCI archive.


In [ ]:
from pathlib import Path
import sys
import urllib.request
import zipfile
import shutil

root = Path.cwd()
while not (root / "src" / "higgs_event_lab").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src"))

raw_path = root / "data" / "raw" / "HIGGS.csv.gz"
processed_path = root / "data" / "processed" / "higgs_sample_250k.csv.gz"
raw_path.parent.mkdir(parents=True, exist_ok=True)
processed_path.parent.mkdir(parents=True, exist_ok=True)

url = "https://archive.ics.uci.edu/static/public/280/higgs.zip"

if not raw_path.exists():
    print("Downloading HIGGS from UCI...")
    zip_path = raw_path.parent / "higgs_download.zip"
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path) as archive:
        gz_name = [name for name in archive.namelist() if name.endswith(".csv.gz")][0]
        with archive.open(gz_name) as source, open(raw_path, "wb") as target:
            shutil.copyfileobj(source, target)
    zip_path.unlink()

if not processed_path.exists() and raw_path.exists():
    from higgs_event_lab.data import extract_sample
    extract_sample(source_path=raw_path, output_path=processed_path, n_rows=250000, compare_later_chunk=False)
    print("Processed sample ready:", processed_path)


## 1. Environment and Package Initialization

**Question:** Which libraries and modules are required to run the deep learning benchmark?

**Why it matters:** To guarantee strict experimental reproducibility and modular design, all model layers, training loops, and data loading contracts reside in tested package modules rather than loose notebook script blocks.

**Explanation:** We import PyTorch for building the neural network, alongside NumPy and Pandas for tensor operations. We also import our project modules `features`, `protocol`, and `neural` to enforce schema constraints and split isolation.

**Expected output:** Confirmation of the PyTorch version and available device acceleration.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from higgs_event_lab.features import FEATURE_GROUPS, get_feature_columns
from higgs_event_lab.protocol import SplitConfig, create_stratified_splits
from higgs_event_lab.neural import MLPConfig, build_mlp, train_mlp, write_run_artifacts

print("PyTorch version:", torch.__version__)
print("CUDA accelerator available:", torch.cuda.is_available())

## 2. Multi-Layer Perceptron Architecture and Regularization

**Question:** How is the neural network structured to prevent overfitting and ensure fast, stable convergence?

**Why it matters:** Unregularized deep networks easily overfit high-dimensional collision data. Batch normalization stabilizes internal covariate shift during gradient updates, while dropout prevents co-adaptation of hidden features.

**Explanation:** The compact MLP architecture applies sequential hidden blocks containing a linear projection, one-dimensional batch normalization, ReLU activation, and twenty percent dropout. The final layer outputs an unconstrained binary logit.

**Expected output:** Layer-by-layer architectural printout and parameter counts.

In [ ]:
config = MLPConfig(hidden_sizes=(64, 32), dropout=0.20, learning_rate=1e-3)
sample_model = build_mlp(input_features=28, config=config)
total_params = sum(param.numel() for param in sample_model.parameters())
print(sample_model)
print("Total network parameters:", total_params)

## 3. The 21 Low-Level and 7 High-Level Feature Groups

**Question:** How are the twenty-eight input variables partitioned across kinematic and derived feature groups?

**Why it matters:** The central scientific question of the benchmark is evaluating whether neural networks require human-engineered invariant masses or can learn equivalent or superior decision boundaries directly from raw detector quantities.

**Explanation:** Low-level features comprise momenta, pseudorapidity, azimuthal angles, and b-tagging metrics. High-level features comprise invariant masses calculated from reconstructed decay subsystems.

**Expected output:** Verified counts and column names for the low-level, high-level, and complete feature sets.

In [ ]:
for group_name in ("low_level", "high_level", "all"):
    columns = get_feature_columns(group_name)
    print(f"{group_name:<11}: {len(columns)} features -> {list(columns[:3])}...")

## 4. Reproducible Stratified Splitting Without Data Leakage

**Question:** How do we partition the dataset into training, validation, and test subsets fairly?

**Why it matters:** If feature values or validation data influence model scalers or threshold selection, evaluation metrics become artificially inflated and unreliable.

**Explanation:** We instantiate a deterministic synthetic dataset and apply the shared stratified splitting protocol using fixed seed forty-two. The training set is seventy percent, with fifteen percent reserved for validation and fifteen percent for held-out testing.

**Expected output:** Exact row counts for each disjoint partition.

In [ ]:
rng = np.random.default_rng(42)
sample_count = 200
synthetic_y = rng.integers(0, 2, size=sample_count)
synthetic_X = pd.DataFrame(
    rng.normal(0.0, 1.0, size=(sample_count, 28)),
    columns=get_feature_columns("all")
)
splits = create_stratified_splits(synthetic_y, SplitConfig(seed=42))
print("Training event count:", len(splits.train))
print("Validation event count:", len(splits.validation))
print("Held-out test event count:", len(splits.test))

## 5. Training Across All Three Feature Sets

**Question:** How does model performance vary across low-level, high-level, and combined feature representations?

**Why it matters:** Comparing model metrics across the three feature groups reveals the relative importance of engineered domain features versus raw sensor measurements.

**Explanation:** We train identical MLP architectures across each subset using AdamW with early stopping monitored on validation ROC-AUC. When training halts, the best historical model weights are automatically restored.

**Expected output:** Completed training epoch count and test evaluation metrics for each feature group.

In [ ]:
benchmark_results = {}
for group_name in ("low_level", "high_level", "all"):
    feature_names = get_feature_columns(group_name)
    X_subset = synthetic_X[list(feature_names)]
    cfg = MLPConfig(hidden_sizes=(32, 16), max_epochs=4, batch_size=32, device="cpu", seed=42)
    benchmark_results[group_name] = train_mlp(X_subset, synthetic_y, split=splits, config=cfg)
    metrics = benchmark_results[group_name]["test"]["metrics"]
    print(f"Group: {group_name:<11} | Epochs: {len(benchmark_results[group_name]["history"])} | Test ROC-AUC: {metrics["roc_auc"]:.4f}")

## 6. Artifact Export for Dashboard and Reports

**Question:** How are training histories and test predictions saved for Ronit Singh’s Streamlit visualization dashboard?

**Why it matters:** Hand-offs between modeling and dashboarding must not rely on copy-pasting numbers. Standardized JSON envelopes ensure full provenance tracking and reproducibility.

**Explanation:** We invoke `write_run_artifacts` to save `metrics.json` and `test_predictions.csv` into the local run directory according to schema version `higgs-experiment-v1`.

**Expected output:** Confirmation of exported metadata and test prediction files.

In [ ]:
run_dir = Path("..") / ".." / "outputs" / "demo_mlp_all"
meta_path, pred_path = write_run_artifacts(
    benchmark_results["all"],
    run_dir=run_dir,
    run_id="demo_mlp_all",
    feature_group="all",
    feature_names=list(synthetic_X.columns),
    data_source="synthetic_demonstration_matrix",
    row_count=sample_count,
    config=MLPConfig(hidden_sizes=(32, 16), device="cpu", seed=42),
)
print("Exported metadata path:", meta_path.resolve())
print("Exported prediction path:", pred_path.resolve())